# 🧪 Laboratorio de Arquitecturas · Módulo 2
### Redes Neuronales — Deep Learning · Maestría en Ciencia de Datos · Universidad Santo Tomás

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JotaMao1985/Deep_Learning_Usta/blob/main/notebooks/03-lab-arquitecturas-modulo-2.ipynb)

Este cuaderno es el **complemento ejecutable** del material del Módulo 2: lleva de una **CNN que clasifica imágenes**, pasa por una **LSTM que pronostica una serie temporal** y termina con un **autoencoder que comprime y reconstruye**, reproduciendo en código las ideas de los Capítulos 2.1 a 2.3.

> **Cómo usarlo:** ejecutar las celdas en orden (en Google Colab: *Entorno de ejecución → Ejecutar todas*). Está calibrado para correr de punta a punta en **CPU en ~2–3 minutos** gracias a subconjuntos pequeños y pocas épocas. Conviene cambiar los valores y volver a ejecutar para observar el efecto.

> ⚠️ **Importante:** este laboratorio usa **datos de juguete** (FashionMNIST) y **series sintéticas**, no los datos reales del proyecto (café, clima). **No constituye la solución de la Actividad evaluable · Fase 2 «Centinela»**: su propósito es practicar las arquitecturas que esa actividad exige.

---
**Contenido**
1. Preparación del entorno
2. CNN sobre imágenes · *Cap. 2.1*
3. Secuencias con LSTM · *Cap. 2.2*
4. Autoencoder: comprimir y reconstruir · *Cap. 2.3*
5. Puente a la Actividad (Centinela · Fase 2)

## 1. Preparación del entorno

Se importan las librerías (todas vienen preinstaladas en Colab), se fija una **semilla** para que los resultados sean reproducibles y se detecta el **dispositivo** disponible (CPU, o MPS en un Mac con Apple Silicon).

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 1 · Preparación del entorno
# Qué hace · Importa librerías, fija la semilla, detecta el dispositivo y configura el estilo.
# Por qué  · Una semilla fija hace reproducibles los resultados; la paleta da identidad USTA.

# En Colab estas librerías ya están instaladas. Si faltara alguna, descomentar:
# !pip -q install torch torchvision scikit-learn matplotlib

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset

# Reproducibilidad: misma semilla -> mismos resultados
SEMILLA = 42
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

# Detección del dispositivo: CPU siempre disponible; MPS en Mac con Apple Silicon.
# Este laboratorio está calibrado para correr cómodo en CPU.
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# Paleta USTA para las gráficas
USTA_MORADO, USTA_ROSA, USTA_NAVY = "#3D008D", "#ED1E79", "#001A4D"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

print(f"Entorno listo · PyTorch {torch.__version__} · torchvision {torchvision.__version__}")
print(f"Dispositivo de cálculo: {DEVICE}")

## 2. CNN sobre imágenes · *Cap. 2.1*

Una **Red Neuronal Convolucional (CNN)** procesa imágenes deslizando **filtros aprendibles** sobre la entrada para producir **mapas de características**, en lugar de aplanar los píxeles como haría un MLP. Esto preserva la **estructura espacial** (qué píxeles están cerca de cuáles) y, gracias a los **pesos compartidos**, detecta la misma característica en cualquier posición de la imagen (*invarianza a la traslación*).

La salida de una capa convolucional respeta la fórmula de tamaño:

$$ O = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1 $$

donde $W$ es el tamaño de entrada, $K$ el del filtro, $P$ el *padding* y $S$ el *stride*. Entre capas convolucionales se intercala **pooling** para reducir el tamaño espacial y aportar robustez.

Aquí se entrena una **CNN pequeña** (2 capas convolucionales + 1 densa) sobre **FashionMNIST**, un conjunto de imágenes $28\times28$ en escala de grises con 10 categorías de prendas. Para que corra rápido en CPU se usa un **subconjunto** de 2.000 imágenes de entrenamiento y 500 de prueba.

> 👕 **De la finca al tensor:** FashionMNIST hace de campo de práctica. Clasificar diez tipos de prenda tiene la misma *forma* que el problema de la **rama A (visión)** de la Fase 2 —decidir si una hoja de café está sana o enferma a partir de sus píxeles—, pero con datos de juguete que no regalan la solución.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 2 · Descarga de FashionMNIST y subconjunto pequeño
# Qué hace · Descarga FashionMNIST a ./data y toma 2.000 imágenes de train y 500 de test.
# Por qué  · Un subconjunto reducido permite entrenar en CPU en segundos sin perder la idea.

# Las imágenes se convierten a tensor y se normalizan al rango aproximado [-1, 1].
transformacion = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# download=True descarga una sola vez a la carpeta ./data (queda cacheada).
train_full = torchvision.datasets.FashionMNIST(root="./data", train=True,  download=True, transform=transformacion)
test_full  = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True, transform=transformacion)

# Subconjuntos: 2.000 de entrenamiento y 500 de prueba (índices fijos -> reproducible).
N_TRAIN, N_TEST = 2000, 500
train_ds = Subset(train_full, range(N_TRAIN))
test_ds  = Subset(test_full,  range(N_TEST))

# DataLoaders: cargan los datos por lotes (mini-batches) de 64.
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

CLASES = ["Camiseta", "Pantalón", "Pulóver", "Vestido", "Abrigo",
          "Sandalia", "Camisa", "Tenis", "Bolso", "Botín"]
print(f"Entrenamiento: {N_TRAIN:,} imágenes · Prueba: {N_TEST:,} imágenes".replace(",", "."))
print(f"Cada imagen: 1 canal × 28 × 28 píxeles · {len(CLASES)} categorías")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 3 · Vistazo a los datos
# Qué hace · Dibuja una rejilla de 8 imágenes de entrenamiento con su etiqueta real.
# Por qué  · Conviene mirar los datos antes de modelar: FashionMNIST son prendas en gris 28×28.

fig, axes = plt.subplots(2, 4, figsize=(8, 4.2))
for ax, (img, etiqueta) in zip(axes.ravel(), train_ds):
    ax.imshow(img.squeeze().numpy(), cmap="gray")
    ax.set_title(CLASES[etiqueta], fontsize=10)
    ax.axis("off")
fig.suptitle("Muestras de FashionMNIST (datos de juguete)")
plt.tight_layout(); plt.show()

### Definición de la CNN
La red apila dos bloques **convolución → ReLU → max-pooling** y termina en una capa densa que produce las 10 puntuaciones de clase. Cada `MaxPool2d(2)` reduce el lado a la mitad: $28 \to 14 \to 7$, de modo que tras el segundo bloque cada imagen queda como un volumen $32 \times 7 \times 7$ que se **aplana** antes de la capa de clasificación.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 4 · Definición del modelo: CNN pequeña (nn.Module)
# Qué hace · Construye 2 bloques conv→ReLU→pool y una capa densa final de 10 salidas.
# Por qué  · Es la arquitectura mínima de visión: extrae características y luego clasifica.

class CNNpequena(nn.Module):
    """CNN compacta para FashionMNIST: 2 bloques convolucionales + 1 capa densa."""
    def __init__(self, n_clases=10):
        super().__init__()
        self.extractor = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 1 -> 16 canales, conserva 28×28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28 -> 14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16 -> 32 canales, conserva 14×14
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 14 -> 7
        )
        self.clasificador = nn.Sequential(
            nn.Flatten(),                 # aplana el volumen 32×7×7 en un vector
            nn.Linear(32 * 7 * 7, n_clases),  # capa densa -> puntuaciones de las 10 clases
        )

    def forward(self, x):
        x = self.extractor(x)
        return self.clasificador(x)       # devuelve logits (sin softmax: lo aplica la pérdida)

torch.manual_seed(SEMILLA)
cnn = CNNpequena().to(DEVICE)
n_param = sum(p.numel() for p in cnn.parameters())
print(cnn)
print(f"\nParámetros entrenables: {n_param:,}".replace(",", "."))

### Entrenamiento (1–2 épocas)
El bucle ejecuta los mismos cinco pasos del Módulo 1 —*forward → error → `zero_grad` → `backward` → `step`*—, ahora por **lotes** que entrega el `DataLoader`. La pérdida es **entropía cruzada** (`CrossEntropyLoss`), que internamente aplica *softmax*; por eso la red devuelve *logits* sin activación final.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 5 · Bucle de entrenamiento de la CNN
# Qué hace · Entrena la CNN durante 2 épocas con CrossEntropyLoss y Adam, por mini-batches.
# Por qué  · Es el motor de aprendizaje; con un subconjunto pequeño basta para ver señal.

criterio = nn.CrossEntropyLoss()                       # pérdida multiclase (incluye softmax)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
EPOCAS = 2

for epoca in range(EPOCAS):
    cnn.train()
    perdida_acum = 0.0
    for imagenes, etiquetas in train_loader:
        imagenes, etiquetas = imagenes.to(DEVICE), etiquetas.to(DEVICE)
        logits = cnn(imagenes)            # 1. forward pass
        loss = criterio(logits, etiquetas)  # 2. cálculo del error
        opt.zero_grad()                   # 3. limpiar gradientes acumulados
        loss.backward()                   # 4. backward pass (retropropagación)
        opt.step()                        # 5. paso del optimizador (actualiza pesos)
        perdida_acum += loss.item() * imagenes.size(0)
    print(f"Época {epoca + 1}/{EPOCAS} · pérdida media: {perdida_acum / N_TRAIN:.4f}")

print("Entrenamiento finalizado.")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 6 · Precisión en el conjunto de prueba
# Qué hace · Evalúa la CNN sobre las 500 imágenes de prueba y reporta la exactitud.
# Por qué  · Mide la generalización: el desempeño sobre datos que la red no vio al entrenar.

cnn.eval()
aciertos, total = 0, 0
with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes, etiquetas = imagenes.to(DEVICE), etiquetas.to(DEVICE)
        pred = cnn(imagenes).argmax(dim=1)   # clase con mayor puntuación
        aciertos += (pred == etiquetas).sum().item()
        total += etiquetas.size(0)

precision = 100 * aciertos / total
print(f"Precisión en prueba: {precision:.1f}%   ({aciertos} de {total} aciertos)")

### Rejilla de predicciones
Se muestran ocho imágenes de prueba con la **predicción** de la red. En verde los aciertos, en rosa los errores. Con solo dos épocas y 2.000 imágenes el modelo ya distingue la mayoría de las prendas; los errores típicos confunden categorías parecidas (camisa ↔ pulóver ↔ abrigo).

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 7 · Rejilla de predicciones
# Qué hace · Predice 8 imágenes de prueba y las dibuja con su etiqueta real y la predicha.
# Por qué  · Inspección cualitativa: revela qué clases confunde la red (camisa↔pulóver↔abrigo).

cnn.eval()
fig, axes = plt.subplots(2, 4, figsize=(9, 4.6))
with torch.no_grad():
    for ax, (img, real) in zip(axes.ravel(), test_ds):
        logit = cnn(img.unsqueeze(0).to(DEVICE))
        pred = logit.argmax(dim=1).item()
        correcto = (pred == real)
        color = USTA_NAVY if correcto else USTA_ROSA
        ax.imshow(img.squeeze().numpy(), cmap="gray")
        ax.set_title(f"real: {CLASES[real]}\npred: {CLASES[pred]}", fontsize=9, color=color)
        ax.axis("off")
fig.suptitle("Predicciones de la CNN (navy = acierto · rosa = error)")
plt.tight_layout(); plt.show()

## 3. Secuencias con LSTM · *Cap. 2.2*

En los **datos secuenciales** el orden importa: el valor de hoy depende del de ayer. Una **RNN** introduce un *estado oculto* que actúa como memoria, pero las RNN simples sufren el **desvanecimiento del gradiente** y olvidan dependencias largas. La **LSTM** (Long Short-Term Memory) resuelve esto con un **estado de celda** —una "cinta transportadora"— y tres **compuertas** (olvido, entrada, salida) que regulan qué información se conserva. La clave matemática es la actualización **aditiva** del estado de celda:

$$ C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t $$

que deja fluir el gradiente sin degradarlo a lo largo de muchos pasos.

Aquí se genera una **serie temporal sintética** (suma de senoidales + ruido) y se entrena una LSTM pequeña para el **pronóstico de un paso** ($t \to t{+}1$) a partir de una ventana de los últimos valores.

> 📈 **Por qué una serie sintética:** la serie de abajo imita la estacionalidad de los datos climáticos de la **rama B (secuencias)** de la Fase 2 —ciclos que se repiten con ruido encima—, pero es totalmente artificial: practica la mecánica de la LSTM sin tocar los datos reales del proyecto.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 8 · Serie temporal sintética y ventanas deslizantes
# Qué hace · Crea una señal (2 senoidales + ruido) y la trocea en ventanas (X) -> siguiente valor (y).
# Por qué  · La LSTM aprende a pronosticar t+1 a partir de la secuencia de los últimos pasos.

torch.manual_seed(SEMILLA); np.random.seed(SEMILLA)

# Señal sintética: dos ciclos de distinta frecuencia + ruido gaussiano leve.
t = np.linspace(0, 8 * np.pi, 600)
serie = np.sin(t) + 0.5 * np.sin(2.5 * t) + 0.15 * np.random.randn(len(t))
serie = serie.astype(np.float32)

# Ventaneo: cada muestra es una ventana de LOOKBACK valores; la etiqueta es el valor siguiente.
LOOKBACK = 30
def construir_ventanas(s, lookback):
    X, y = [], []
    for i in range(len(s) - lookback):
        X.append(s[i:i + lookback])
        y.append(s[i + lookback])
    X = np.array(X)[:, :, None]   # forma (N, lookback, 1): (batch, pasos, features)
    return X.astype(np.float32), np.array(y, dtype=np.float32)[:, None]

X, y = construir_ventanas(serie, LOOKBACK)

# División temporal (sin barajar): 80% pasado para entrenar, 20% futuro para probar.
corte = int(0.8 * len(X))
X_tr = torch.tensor(X[:corte]); y_tr = torch.tensor(y[:corte])
X_te = torch.tensor(X[corte:]); y_te = torch.tensor(y[corte:])
print(f"Ventanas de entrenamiento: {len(X_tr)} · de prueba: {len(X_te)} · lookback: {LOOKBACK} pasos")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 9 · Definición del modelo: LSTM pequeña (nn.Module)
# Qué hace · Una capa nn.LSTM seguida de una capa densa que produce el valor pronosticado.
# Por qué  · Es la arquitectura mínima de pronóstico secuencial con memoria a largo plazo.

class PronosticadorLSTM(nn.Module):
    """LSTM de una capa + densa: pronostica el siguiente valor de la serie."""
    def __init__(self, n_entrada=1, n_oculta=32):
        super().__init__()
        self.lstm = nn.LSTM(n_entrada, n_oculta, batch_first=True)  # entrada (batch, pasos, features)
        self.salida = nn.Linear(n_oculta, 1)

    def forward(self, x):
        salidas, _ = self.lstm(x)        # salidas: estado oculto en cada paso temporal
        ultimo = salidas[:, -1, :]       # se toma solo el último paso (resumen de la ventana)
        return self.salida(ultimo)       # valor pronosticado para t+1

torch.manual_seed(SEMILLA)
lstm = PronosticadorLSTM().to(DEVICE)
print(lstm)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 10 · Entrenamiento de la LSTM
# Qué hace · Entrena la LSTM 40 épocas con error cuadrático medio (MSE) y Adam.
# Por qué  · El MSE es la pérdida natural para regresión; pocas épocas bastan en esta serie.

X_tr_d, y_tr_d = X_tr.to(DEVICE), y_tr.to(DEVICE)
criterio_mse = nn.MSELoss()
opt = torch.optim.Adam(lstm.parameters(), lr=1e-2)
EPOCAS = 40
hist = []

for epoca in range(EPOCAS):
    lstm.train()
    pred = lstm(X_tr_d)            # 1. forward
    loss = criterio_mse(pred, y_tr_d)  # 2. error
    opt.zero_grad()               # 3. limpiar gradientes
    loss.backward()               # 4. backward
    opt.step()                    # 5. paso del optimizador
    hist.append(loss.item())
    if (epoca + 1) % 10 == 0:
        print(f"Época {epoca + 1:>2}/{EPOCAS} · pérdida MSE: {loss.item():.4f}")

print(f"Pérdida final de entrenamiento: {hist[-1]:.4f}")

### Predicción vs. real
Se evalúa la LSTM sobre el tramo de prueba (el "futuro" que no vio al entrenar) y se superpone el pronóstico con la serie real. Un buen ajuste sigue la forma de la onda; las pequeñas diferencias corresponden al ruido, que es por definición impredecible.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 11 · Predicción vs. real y curva de pérdida
# Qué hace · Grafica el pronóstico de la LSTM sobre el tramo de prueba y la curva de pérdida.
# Por qué  · Verifica visualmente que la LSTM captura la forma de la onda, no solo el promedio.

lstm.eval()
with torch.no_grad():
    pred_te = lstm(X_te.to(DEVICE)).cpu().numpy().ravel()
real_te = y_te.numpy().ravel()
mse_te = np.mean((pred_te - real_te) ** 2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(real_te, color=USTA_NAVY, lw=2, label="Real")
axes[0].plot(pred_te, color=USTA_ROSA, lw=2, ls="--", label="Pronóstico LSTM")
axes[0].set_xlabel("Paso temporal (tramo de prueba)"); axes[0].set_ylabel("Valor")
axes[0].set_title(f"Pronóstico de 1 paso · MSE prueba = {mse_te:.4f}"); axes[0].legend()

axes[1].plot(hist, color=USTA_MORADO, lw=2)
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Pérdida (MSE)")
axes[1].set_title("Curva de pérdida (entrenamiento)")
plt.tight_layout(); plt.show()

## 4. Autoencoder: comprimir y reconstruir · *Cap. 2.3*

Un **autoencoder (AE)** es una red no supervisada que aprende a **comprimir** la entrada en un vector pequeño (el *código* o espacio latente) y luego **reconstruirla** desde esa compresión. Se compone de un **encoder** que reduce la dimensión y un **decoder** que la expande de vuelta:

$$ z = f(x), \qquad \hat{x} = g(z), \qquad \mathcal{L} = \lVert x - \hat{x} \rVert^2 $$

El **cuello de botella** —el código intermedio, mucho más pequeño que la entrada— obliga a la red a quedarse solo con la información esencial y descartar el ruido. Esto tiene una aplicación directa: un AE entrenado con datos *normales* produce un **error de reconstrucción alto** ante datos anómalos, base de la **detección de anomalías**.

Aquí se entrena un AE sobre el mismo subconjunto de FashionMNIST y se comparan **originales vs. reconstrucciones**.

> 🔁 **Conexión con la Fase 2:** la reconstrucción y el error que de ella se deriva son una herramienta complementaria de la **rama A (visión)** —por ejemplo, señalar hojas con un aspecto fuera de lo común—. El autocodificador del Cap. 2.3 es la pieza que lo hace posible.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 12 · Definición del autoencoder (MLP)
# Qué hace · Encoder 784->128->32 y decoder simétrico 32->128->784 con salida en [-1, 1].
# Por qué  · El cuello de botella de 32 valores fuerza a la red a comprimir lo esencial.

class Autoencoder(nn.Module):
    """Autoencoder denso: comprime 784 -> 32 (cuello de botella) y reconstruye."""
    def __init__(self, dim_entrada=28 * 28, dim_codigo=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(dim_entrada, 128), nn.ReLU(),
            nn.Linear(128, dim_codigo),  nn.ReLU(),   # código latente comprimido
        )
        self.decoder = nn.Sequential(
            nn.Linear(dim_codigo, 128), nn.ReLU(),
            nn.Linear(128, dim_entrada),
            nn.Tanh(),                                # salida en [-1, 1], igual que la entrada normalizada
        )

    def forward(self, x):
        z = self.encoder(x)        # compresión
        x_rec = self.decoder(z)    # reconstrucción (vector plano de 784)
        return x_rec.view(-1, 1, 28, 28)  # se devuelve a la forma de imagen

torch.manual_seed(SEMILLA)
ae = Autoencoder().to(DEVICE)
print(ae)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 13 · Entrenamiento del autoencoder
# Qué hace · Entrena el AE 3 épocas con MSE entre la imagen original y su reconstrucción.
# Por qué  · Es aprendizaje NO supervisado: la "etiqueta" es la propia imagen de entrada.

criterio_mse = nn.MSELoss()
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
EPOCAS = 3

for epoca in range(EPOCAS):
    ae.train()
    perdida_acum = 0.0
    for imagenes, _ in train_loader:          # las etiquetas se ignoran (no supervisado)
        imagenes = imagenes.to(DEVICE)
        recon = ae(imagenes)                  # 1. forward: comprime y reconstruye
        loss = criterio_mse(recon, imagenes)  # 2. error de reconstrucción
        opt.zero_grad()                       # 3. limpiar gradientes
        loss.backward()                       # 4. backward
        opt.step()                            # 5. paso del optimizador
        perdida_acum += loss.item() * imagenes.size(0)
    print(f"Época {epoca + 1}/{EPOCAS} · MSE de reconstrucción: {perdida_acum / N_TRAIN:.4f}")

print("Autoencoder entrenado.")

### Original vs. reconstrucción
La fila superior muestra imágenes de prueba originales; la inferior, lo que el AE logra reconstruir tras pasar por el cuello de botella de 32 valores. Las reconstrucciones aparecen algo borrosas —es el precio de comprimir 784 píxeles en 32 números—, pero conservan la silueta y el tipo de prenda: prueba de que el código latente retuvo lo esencial.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 14 · Reconstrucciones: original vs. reconstruida
# Qué hace · Reconstruye 8 imágenes de prueba y las dibuja sobre su versión original.
# Por qué  · Inspección cualitativa del autoencoder: ¿qué se conserva tras comprimir a 32 valores?

ae.eval()
muestras = torch.stack([test_ds[i][0] for i in range(8)]).to(DEVICE)
with torch.no_grad():
    recon = ae(muestras).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for col in range(8):
    axes[0, col].imshow(muestras[col].cpu().squeeze().numpy(), cmap="gray"); axes[0, col].axis("off")
    axes[1, col].imshow(recon[col].squeeze().numpy(),          cmap="gray"); axes[1, col].axis("off")
axes[0, 0].set_ylabel("Original",      rotation=0, ha="right", va="center")
axes[1, 0].set_ylabel("Reconstruida",  rotation=0, ha="right", va="center")
fig.suptitle("Autoencoder · fila superior: original · fila inferior: reconstrucción")
plt.tight_layout(); plt.show()

## 5. Puente a la Actividad · Centinela · Fase 2

Lo practicado aquí es **exactamente** el andamiaje técnico de la actividad evaluable de la Fase 2 — aplicado a datos de juguete y sintéticos. La Fase 2 construye un sistema **multimodal** sobre tres ramas; la tabla traduce cada habilidad ensayada en el laboratorio a su lugar en el proyecto:

| En este laboratorio (datos toy/sintéticos) | En la Fase 2 «Centinela» (datos reales del estudiante) |
|---|---|
| CNN sobre FashionMNIST (Cap. 2.1) | **Rama A (visión):** clasificación de imágenes reales con **Transfer Learning / Fine-Tuning** sobre una CNN pre-entrenada |
| `CrossEntropyLoss` + Adam + bucle por lotes | mismo motor de entrenamiento, con `DataLoader`, métricas por clase y matriz de confusión |
| LSTM sobre serie sintética (Cap. 2.2) | **Rama B (secuencias):** pronóstico de una **serie temporal real** (clima/finanzas/logística) con LSTM/GRU y una RNN simple de contraste |
| ventanas deslizantes + división temporal | mismo preprocesamiento, respetando el orden temporal y justificando el *lookback* |
| autoencoder: comprimir y reconstruir (Cap. 2.3) | técnica complementaria de la rama visual (reconstrucción / detección de anomalías) |
| dos modalidades practicadas por separado | **Rama C (fusión):** combinar la rama de imagen y la de serie en un **único predictor multimodal** |

> ⚠️ **Recordatorio:** este cuaderno **no** entrega la solución. La Fase 2 exige aplicar todo esto a **datos reales elegidos por el estudiante** (no de Kaggle), justificar las arquitecturas, fusionar las dos modalidades, documentar el **uso responsable de IA** (bitácora + auditoría) y sustentar el trabajo oralmente.

> 📝 *Nota de escalado:* en este laboratorio la LSTM y el AE se entrenaron con pocos datos y pocas épocas para correr en CPU. Con conjuntos reales conviene aumentar las épocas, usar un dispositivo acelerado (GPU/MPS), validar con un conjunto separado y vigilar el **sobreajuste** con las técnicas del Módulo 1 (Dropout, *weight decay*, *early stopping*).

### 🔗 Relacionado
- [Material interactivo del Módulo 2](https://jotamao1985.github.io/Deep_Learning_Usta/) (Capítulos 2.1–2.3) y la actividad navegable [Centinela · Fase 2](https://jotamao1985.github.io/Deep_Learning_Usta/21_Modulo_2_Actividad_Fase2_Centinela.html).
- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*, Cap. 9 (CNN), 10 (redes recurrentes) y 14 (autoencoders). MIT Press.
- [PyTorch · tutoriales de visión y secuencias](https://pytorch.org/tutorials/) — referencia oficial complementaria.